In [1]:
# 1차 union 방법 형이랑 얘기했던 방법
# 문제점: union으로 하면 타일 수가 너무 많음 점수가 낮음 "그 만큼 gene 별로 분표가 많이 다르다는 방증"
import numpy as np, torch, torch.nn.functional as F, pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os

device = 'cuda'
GENE_LIST = '/project_antwerp/hbae/data/0317_hvg_2000_list.txt'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_03'
TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_03'
OUT_DIR   = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/alpha_search'
os.makedirs(OUT_DIR, exist_ok=True)

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]
G = len(common_genes)  # 1968

train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
train_expr = torch.tensor(
    np.load(f'{FT_EMB}/train_exprs.npy'), dtype=torch.float32, device=device)

matched = [(row['slide_id'], row[bulk_cols].values.astype(float))
           for _, row in ref_df.iterrows()
           if os.path.exists(f'{TCGA_EMB}/{row["slide_id"]}.npy')]
print(f'Slides: {len(matched)}, Genes: {G}')

# ── Step 1: 모든 슬라이드의 tile_preds 미리 계산 ──────────
print('Computing tile predictions...')
slide_tile_preds = []  # list of (T_i, G) arrays
slide_bulks      = []  # list of (G,) arrays

for sid, bulk in matched:
    embs = F.normalize(torch.tensor(
        np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)

    with torch.no_grad():
        sim        = torch.clamp(embs @ train_embs.T, min=0)
        weights    = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
        tile_preds = (weights @ train_expr).cpu().numpy()  # (T, 2000)

    slide_tile_preds.append(tile_preds[:, common_idx])  # (T, 1968)
    slide_bulks.append(bulk)
    del embs
torch.cuda.empty_cache()

S = len(slide_tile_preds)
bulk_arr = np.array(slide_bulks)  # (S, G)
print(f'Done. Total slides: {S}')

# ── Step 2: alpha search ──────────────────────────────────
alphas = np.arange(0.01, 0.11, 0.01)  # 0.01 ~ 0.10

gene_pcc_results  = []  # (n_alpha,) mean gene-wise PCC
slide_pcc_results = []  # (n_alpha,) mean slide-wise PCC
union_size_results = [] # (n_alpha,) avg union tile count

print(f'\nalpha search: {alphas}')
print(f'{"alpha":>6} | {"gene_mean":>10} | {"slide_mean":>11} | {"avg_union":>10}')
print('-' * 45)

for alpha in alphas:
    pred_pseudo_bulks = []  # (S, G)
    union_sizes       = []

    for i, tile_preds in enumerate(slide_tile_preds):
        T, G_local = tile_preds.shape
        k = max(1, int(np.ceil(T * alpha)))  # 상위 alpha% tile 수

        # 각 gene별로 상위 k개 tile index 선택
        # tile_preds: (T, G) → 각 gene별 top-k tile
        # argsort along axis=0 (tile axis), 내림차순
        top_k_indices = np.argpartition(tile_preds, -k, axis=0)[-k:]  # (k, G)

        # Union: 모든 gene의 top-k tile index 합집합
        union_idx = np.unique(top_k_indices.flatten())
        union_sizes.append(len(union_idx))

        # Union tile들의 예측값 합산 → pseudo-bulk
        pseudo_bulk = tile_preds[union_idx].sum(axis=0)  # (G,)
        pred_pseudo_bulks.append(pseudo_bulk)

    pred_arr = np.array(pred_pseudo_bulks)  # (S, G)

    # Gene-wise PCC
    gene_pccs = [pearsonr(pred_arr[:,j], bulk_arr[:,j])[0]
                 for j in range(G)
                 if pred_arr[:,j].std() > 1e-8 and bulk_arr[:,j].std() > 1e-8]

    # Slide-wise PCC
    slide_pccs = [pearsonr(pred_arr[i], bulk_arr[i])[0]
                  for i in range(S)
                  if pred_arr[i].std() > 1e-8 and bulk_arr[i].std() > 1e-8]

    g_mean = np.array(gene_pccs).mean()
    s_mean = np.array(slide_pccs).mean()
    u_mean = np.mean(union_sizes)

    gene_pcc_results.append(g_mean)
    slide_pcc_results.append(s_mean)
    union_size_results.append(u_mean)

    print(f'{alpha:>6.2f} | {g_mean:>10.4f} | {s_mean:>11.4f} | {u_mean:>10.1f}')

# ── Step 3: 최적 alpha 찾기 ───────────────────────────────
best_idx   = np.argmax(gene_pcc_results)
best_alpha = alphas[best_idx]
best_pcc   = gene_pcc_results[best_idx]
print(f'\n최적 alpha: {best_alpha:.2f}  →  Gene-wise PCC: {best_pcc:.4f}')

# ── Step 4: 시각화 ────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Per-Gene Top-α% Tile Union Sum: α Search (fold_03)', fontsize=13)

# Gene-wise PCC
axes[0].plot(alphas, gene_pcc_results, 'b-o', linewidth=2, markersize=6)
axes[0].axvline(x=best_alpha, color='red', linestyle='--', alpha=0.7)
axes[0].axhline(y=best_pcc, color='red', linestyle=':', alpha=0.5)
axes[0].scatter([best_alpha], [best_pcc], color='red', s=100, zorder=5,
                label=f'Best α={best_alpha:.2f}\nPCC={best_pcc:.4f}')
axes[0].set_xlabel('α (top tile ratio per gene)', fontsize=11)
axes[0].set_ylabel('Gene-wise PCC (mean)', fontsize=11)
axes[0].set_title('Gene-wise PCC vs α')
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)
axes[0].set_xticks(alphas)
axes[0].set_xticklabels([f'{a:.0%}' for a in alphas], rotation=45)

# Slide-wise PCC
axes[1].plot(alphas, slide_pcc_results, 'g-o', linewidth=2, markersize=6)
best_slide_idx = np.argmax(slide_pcc_results)
axes[1].scatter([alphas[best_slide_idx]], [slide_pcc_results[best_slide_idx]],
                color='red', s=100, zorder=5,
                label=f'Best α={alphas[best_slide_idx]:.2f}\nPCC={slide_pcc_results[best_slide_idx]:.4f}')
axes[1].set_xlabel('α (top tile ratio per gene)', fontsize=11)
axes[1].set_ylabel('Slide-wise PCC (mean)', fontsize=11)
axes[1].set_title('Slide-wise PCC vs α')
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)
axes[1].set_xticks(alphas)
axes[1].set_xticklabels([f'{a:.0%}' for a in alphas], rotation=45)

# Union size
axes[2].plot(alphas, union_size_results, 'm-o', linewidth=2, markersize=6)
axes[2].set_xlabel('α', fontsize=11)
axes[2].set_ylabel('Avg union tile count', fontsize=11)
axes[2].set_title('Avg Union Tile Count vs α')
axes[2].grid(alpha=0.3)
axes[2].set_xticks(alphas)
axes[2].set_xticklabels([f'{a:.0%}' for a in alphas], rotation=45)

plt.tight_layout()
save_path = f'{OUT_DIR}/alpha_search_fold03.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Saved: {save_path}')
print('Done!')


Slides: 331, Genes: 1968
Computing tile predictions...
Done. Total slides: 331

alpha search: [0.01 0.02 0.03 0.04 0.05 0.06 0.07 0.08 0.09 0.1 ]
 alpha |  gene_mean |  slide_mean |  avg_union
---------------------------------------------
  0.01 |    -0.0060 |      0.7238 |     2053.2
  0.02 |    -0.0057 |      0.7238 |     2853.4
  0.03 |    -0.0053 |      0.7237 |     3333.3
  0.04 |    -0.0051 |      0.7237 |     3646.5
  0.05 |    -0.0050 |      0.7237 |     3859.7
  0.06 |    -0.0048 |      0.7237 |     4008.8
  0.07 |    -0.0048 |      0.7237 |     4116.6
  0.08 |    -0.0048 |      0.7237 |     4193.9
  0.09 |    -0.0047 |      0.7237 |     4249.8
  0.10 |    -0.0048 |      0.7237 |     4289.5

최적 alpha: 0.09  →  Gene-wise PCC: -0.0047
Saved: /project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/alpha_search/alpha_search_fold03.png
Done!


In [2]:
# 2차 intersection 방법 형이랑 얘기했던 방법
# 문제점: intersection으로 하면 타일 수가 공집합 나옴 (0개)  "그 만큼 gene 별로 분표가 많이 다르다는 방증"
import numpy as np, torch, torch.nn.functional as F, pandas as pd
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import os

device = 'cuda'
GENE_LIST = '/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_03'
TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_03'
OUT_DIR   = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/alpha_search'
os.makedirs(OUT_DIR, exist_ok=True)

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]
G = len(common_genes)

train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
train_expr = torch.tensor(
    np.load(f'{FT_EMB}/train_exprs.npy'), dtype=torch.float32, device=device)

matched = [(row['slide_id'], row[bulk_cols].values.astype(float))
           for _, row in ref_df.iterrows()
           if os.path.exists(f'{TCGA_EMB}/{row["slide_id"]}.npy')]
print(f'Slides: {len(matched)}, Genes: {G}')

# ── Step 1: tile predictions 미리 계산 ───────────────────
print('Computing tile predictions...')
slide_tile_preds = []
slide_bulks      = []

for sid, bulk in matched:
    embs = F.normalize(torch.tensor(
        np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)
    with torch.no_grad():
        sim        = torch.clamp(embs @ train_embs.T, min=0)
        weights    = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
        tile_preds = (weights @ train_expr).cpu().numpy()
    slide_tile_preds.append(tile_preds[:, common_idx])
    slide_bulks.append(bulk)
    del embs
torch.cuda.empty_cache()

S        = len(slide_tile_preds)
bulk_arr = np.array(slide_bulks)
print(f'Done. Slides: {S}')

alphas = np.arange(0.01, 0.11, 0.01)

# ── 결과 저장 ─────────────────────────────────────────────
results = {
    'intersection': {'gene': [], 'slide': [], 'union_size': []},
    'vote':         {'gene': [], 'slide': [], 'union_size': []},
}

def calc_pcc(pred_arr, bulk_arr):
    gene_pccs = [pearsonr(pred_arr[:,j], bulk_arr[:,j])[0]
                 for j in range(pred_arr.shape[1])
                 if pred_arr[:,j].std() > 1e-8 and bulk_arr[:,j].std() > 1e-8]
    slide_pccs = [pearsonr(pred_arr[i], bulk_arr[i])[0]
                  for i in range(pred_arr.shape[0])
                  if pred_arr[i].std() > 1e-8 and bulk_arr[i].std() > 1e-8]
    return np.array(gene_pccs).mean(), np.array(slide_pccs).mean()

# ── 선택 1: Intersection ──────────────────────────────────
print('\n=== 선택 1: Intersection ===')
print(f'{"alpha":>6} | {"gene_mean":>10} | {"slide_mean":>11} | {"avg_size":>10}')
print('-' * 45)

for alpha in alphas:
    preds, sizes = [], []
    for tile_preds in slide_tile_preds:
        T = len(tile_preds)
        k = max(1, int(np.ceil(T * alpha)))

        # gene별 top-k tile index
        top_k_indices = np.argpartition(tile_preds, -k, axis=0)[-k:]  # (k, G)

        # Intersection: 모든 gene에서 공통으로 top-k에 든 tile
        sets = [set(top_k_indices[:, g]) for g in range(G)]
        intersection_idx = np.array(list(sets[0].intersection(*sets[1:])))

        if len(intersection_idx) == 0:
            # intersection이 비면 전체 평균으로 fallback
            pseudo_bulk = tile_preds.sum(axis=0)
            sizes.append(0)
        else:
            pseudo_bulk = tile_preds[intersection_idx].sum(axis=0)
            sizes.append(len(intersection_idx))

        preds.append(pseudo_bulk)

    pred_arr  = np.array(preds)
    g_mean, s_mean = calc_pcc(pred_arr, bulk_arr)
    results['intersection']['gene'].append(g_mean)
    results['intersection']['slide'].append(s_mean)
    results['intersection']['union_size'].append(np.mean(sizes))
    print(f'{alpha:>6.2f} | {g_mean:>10.4f} | {s_mean:>11.4f} | {np.mean(sizes):>10.1f}')

# ── 선택 3: Vote 기반 ─────────────────────────────────────
print('\n=== 선택 3: Vote 기반 (상위 K_vote tile) ===')
# vote 기반은 alpha로 top-k per gene 결정 후
# vote score 높은 상위 K_final tile 선택
# K_final = 100 (이전 실험에서 최적)
K_finals = [50, 100, 200, 300]

for K_final in K_finals:
    print(f'\n  K_final={K_final}:')
    print(f'  {"alpha":>6} | {"gene_mean":>10} | {"slide_mean":>11} | {"avg_size":>10}')
    print('  ' + '-' * 45)

    gene_list_kf, slide_list_kf, size_list_kf = [], [], []

    for alpha in alphas:
        preds, sizes = [], []
        for tile_preds in slide_tile_preds:
            T = len(tile_preds)
            k = max(1, int(np.ceil(T * alpha)))

            # gene별 top-k tile에 vote
            top_k_indices = np.argpartition(tile_preds, -k, axis=0)[-k:]  # (k, G)

            # 각 tile의 vote 수 계산
            vote_counts = np.zeros(T, dtype=int)
            for g in range(G):
                vote_counts[top_k_indices[:, g]] + = 1

            # vote 상위 K_final tile 선택
            k_sel = min(K_final, T)
            selected_idx = np.argsort(vote_counts)[::-1][:k_sel]

            pseudo_bulk = tile_preds[selected_idx].sum(axis=0)
            preds.append(pseudo_bulk)
            sizes.append(len(selected_idx))

        pred_arr  = np.array(preds)
        g_mean, s_mean = calc_pcc(pred_arr, bulk_arr)
        gene_list_kf.append(g_mean)
        slide_list_kf.append(s_mean)
        size_list_kf.append(np.mean(sizes))
        print(f'  {alpha:>6.2f} | {g_mean:>10.4f} | {s_mean:>11.4f} | {np.mean(sizes):>10.1f}')

    results[f'vote_K{K_final}'] = {
        'gene': gene_list_kf,
        'slide': slide_list_kf,
        'union_size': size_list_kf
    }

# ── 시각화 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Per-Gene Top-α% Tile Selection: Intersection vs Vote (fold_03)', fontsize=13)

colors = {'intersection': 'red',
          'vote_K50': '#2ecc71', 'vote_K100': '#3498db',
          'vote_K200': '#e67e22', 'vote_K300': '#9b59b6'}
labels = {'intersection': 'Intersection',
          'vote_K50': 'Vote K=50', 'vote_K100': 'Vote K=100',
          'vote_K200': 'Vote K=200', 'vote_K300': 'Vote K=300'}

for key in ['intersection', 'vote_K50', 'vote_K100', 'vote_K200', 'vote_K300']:
    if key not in results: continue
    axes[0].plot(alphas, results[key]['gene'], '-o', color=colors[key],
                 label=labels[key], linewidth=2, markersize=5)
    axes[1].plot(alphas, results[key]['slide'], '-o', color=colors[key],
                 label=labels[key], linewidth=2, markersize=5)

# 이전 실험 baseline (tile-wise PCC K=100)
axes[0].axhline(y=0.3510, color='black', linestyle='--', alpha=0.6, label='Baseline (tile PCC K=100)')
axes[1].axhline(y=0.7267, color='black', linestyle='--', alpha=0.6, label='Baseline (tile PCC K=100)')

for ax, title, ylabel in zip(axes,
    ['Gene-wise PCC vs α', 'Slide-wise PCC vs α'],
    ['Gene-wise PCC (mean)', 'Slide-wise PCC (mean)']):
    ax.set_xlabel('α (top tile ratio per gene)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    ax.set_xticks(alphas)
    ax.set_xticklabels([f'{a:.0%}' for a in alphas], rotation=45)

plt.tight_layout()
save_path = f'{OUT_DIR}/intersection_vs_vote_fold03.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved: {save_path}')
print('Done!')


Slides: 331, Genes: 1968
Computing tile predictions...
Done. Slides: 331

=== 선택 1: Intersection ===
 alpha |  gene_mean |  slide_mean |   avg_size
---------------------------------------------
  0.01 |    -0.0058 |      0.7193 |        0.0
  0.02 |    -0.0058 |      0.7193 |        0.0
  0.03 |    -0.0058 |      0.7193 |        0.0
  0.04 |    -0.0058 |      0.7193 |        0.0
  0.05 |    -0.0058 |      0.7193 |        0.0
  0.06 |    -0.0058 |      0.7193 |        0.0
  0.07 |    -0.0058 |      0.7193 |        0.0
  0.08 |    -0.0058 |      0.7193 |        0.0
  0.09 |    -0.0058 |      0.7193 |        0.0
  0.10 |    -0.0058 |      0.7193 |        0.0

=== 선택 3: Vote 기반 (상위 K_vote tile) ===

  K_final=50:
   alpha |  gene_mean |  slide_mean |   avg_size
  ---------------------------------------------
    0.01 |     0.0199 |      0.7179 |       50.0
    0.02 |     0.0217 |      0.7177 |       50.0
    0.03 |     0.0251 |      0.7177 |       50.0
    0.04 |     0.0289 |      0.7178 |

In [ ]:

import numpy as np, torch, torch.nn.functional as F, pandas as pd
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import os

device = 'cuda'
GENE_LIST = '/project_antwerp/hbae/data/0317_hvg_2000_list.txt'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_03'
TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_03'
OUT_DIR   = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/metric_comparison'
os.makedirs(OUT_DIR, exist_ok=True)

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]
G = len(common_genes)

train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
train_expr = torch.tensor(
    np.load(f'{FT_EMB}/train_exprs.npy'), dtype=torch.float32, device=device)

matched = [(row['slide_id'], row[bulk_cols].values.astype(float))
           for _, row in ref_df.iterrows()
           if os.path.exists(f'{TCGA_EMB}/{row["slide_id"]}.npy')]
print(f'Slides: {len(matched)}, Genes: {G}')

# ── tile predictions 미리 계산 ────────────────────────────
print('Computing tile predictions...')
slide_tile_preds, slide_bulks = [], []

for sid, bulk in matched:
    embs = F.normalize(torch.tensor(
        np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)
    with torch.no_grad():
        sim        = torch.clamp(embs @ train_embs.T, min=0)
        weights    = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
        tile_preds = (weights @ train_expr).cpu().numpy()
    slide_tile_preds.append(tile_preds[:, common_idx])
    slide_bulks.append(bulk)
    del embs
torch.cuda.empty_cache()

S        = len(slide_tile_preds)
bulk_arr = np.array(slide_bulks)
print(f'Done. Slides: {S}')

# ── PCC 계산 함수 ─────────────────────────────────────────
def calc_pcc(pred_arr, bulk_arr):
    gene_pccs = [pearsonr(pred_arr[:,j], bulk_arr[:,j])[0]
                 for j in range(pred_arr.shape[1])
                 if pred_arr[:,j].std() > 1e-8 and bulk_arr[:,j].std() > 1e-8]
    slide_pccs = [pearsonr(pred_arr[i], bulk_arr[i])[0]
                  for i in range(pred_arr.shape[0])
                  if pred_arr[i].std() > 1e-8 and bulk_arr[i].std() > 1e-8]
    return np.array(gene_pccs).mean(), np.array(slide_pccs).mean()

Ks = [50, 100, 200, 300, 500, 'all']

# ── 4가지 ranking metric 비교 ─────────────────────────────
methods = {
    'PCC':            [],
    'Spearman':       [],
    'Cosine':         [],
    'Weighted_PCC':   [],  # bulk variance로 gene별 가중치
}

print('\n' + '='*70)
print('Ranking metric 비교 (fold_03)')
print('='*70)

for method_name in methods.keys():
    print(f'\n[{method_name}]')
    print(f'{"K":>6} | {"Gene-wise mean":>14} | {"Slide-wise mean":>15}')
    print('-' * 42)

    # bulk variance (weighted PCC용)
    bulk_var = bulk_arr.var(axis=0)  # (G,)
    bulk_var_norm = bulk_var / bulk_var.sum()

    for K in Ks:
        preds = []
        for tile_preds, bulk in zip(slide_tile_preds, slide_bulks):
            T = len(tile_preds)

            # tile별 score 계산
            if method_name == 'PCC':
                scores = np.array([
                    pearsonr(tile_preds[i], bulk)[0]
                    if tile_preds[i].std() > 1e-8 else -999
                    for i in range(T)
                ])

            elif method_name == 'Spearman':
                scores = np.array([
                    spearmanr(tile_preds[i], bulk)[0]
                    if tile_preds[i].std() > 1e-8 else -999
                    for i in range(T)
                ])

            elif method_name == 'Cosine':
                tile_norm = F.normalize(
                    torch.tensor(tile_preds, dtype=torch.float32, device=device), dim=-1)
                bulk_norm = F.normalize(
                    torch.tensor(bulk, dtype=torch.float32, device=device).unsqueeze(0), dim=-1)
                scores = (tile_norm @ bulk_norm.T).squeeze().cpu().numpy()

            elif method_name == 'Weighted_PCC':
                # gene별 bulk variance로 가중치 부여
                scores = np.array([
                    np.sum(bulk_var_norm * np.array([
                        pearsonr(tile_preds[i:i+1, g].flatten(),
                                 bulk[g:g+1].flatten())[0]
                        if tile_preds[i].std() > 1e-8 else 0
                        for g in range(G)
                    ]))
                    for i in range(T)
                ])

            # top K 선택 후 sum
            if K == 'all':
                selected = tile_preds
            else:
                k = min(K, T)
                valid_mask = scores > -999
                valid_idx  = np.where(valid_mask)[0]
                top_idx    = valid_idx[np.argsort(scores[valid_mask])[::-1][:k]]
                selected   = tile_preds[top_idx]

            preds.append(selected.sum(axis=0))

        pred_arr  = np.array(preds)
        g_mean, s_mean = calc_pcc(pred_arr, bulk_arr)
        methods[method_name].append((K, g_mean, s_mean))
        print(f'{str(K):>6} | {g_mean:>14.4f} | {s_mean:>15.4f}')

# ── 최종 비교 표 ──────────────────────────────────────────
print('\n' + '='*70)
print('최종 비교 (K=100 기준)')
print('='*70)
print(f'{"방법":<15} | {"Gene-wise mean":>14} | {"Slide-wise mean":>15}')
print('-' * 50)
for method_name, result_list in methods.items():
    r = {k: (g, s) for k, g, s in result_list}
    print(f'{method_name:<15} | {r[100][0]:>14.4f} | {r[100][1]:>15.4f}')

# ── 시각화 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Tile Ranking Metric Comparison: Gene-wise PCC (fold_03)', fontsize=13)

colors = {'PCC': '#2ecc71', 'Spearman': '#3498db',
          'Cosine': '#e67e22', 'Weighted_PCC': '#9b59b6'}

K_nums = [50, 100, 200, 300, 500]

for method_name, result_list in methods.items():
    r       = {k: (g, s) for k, g, s in result_list}
    g_vals  = [r[k][0] for k in K_nums]
    s_vals  = [r[k][1] for k in K_nums]
    axes[0].plot(K_nums, g_vals, '-o', color=colors[method_name],
                 label=method_name, linewidth=2, markersize=6)
    axes[1].plot(K_nums, s_vals, '-o', color=colors[method_name],
                 label=method_name, linewidth=2, markersize=6)

for ax, title, ylabel in zip(axes,
    ['Gene-wise PCC vs K', 'Slide-wise PCC vs K'],
    ['Gene-wise PCC (mean)', 'Slide-wise PCC (mean)']):
    ax.set_xlabel('K (top tile count)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
save_path = f'{OUT_DIR}/metric_comparison_fold03.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved: {save_path}')
print('Done!')


In [ ]:
python3 << 'EOF'
import numpy as np, torch, torch.nn.functional as F, pandas as pd
from scipy.stats import pearsonr, spearmanr
import matplotlib.pyplot as plt
import os

device = 'cuda'
GENE_LIST = '/project_antwerp/hbae/data/0317_hvg_2000_list.txt'
REF_FILE  = '/project_antwerp/hbae/ref_file.csv'
FT_EMB    = '/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_03'
TCGA_EMB  = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/TCGA_embeddings/fold_03'
OUT_DIR   = '/project_antwerp/hbae/Loki_output/TCGA_bulk_prediction/metric_comparison'
os.makedirs(OUT_DIR, exist_ok=True)

with open(GENE_LIST) as f:
    gene_list = [l.strip() for l in f if l.strip()]

ref_df = pd.read_csv(REF_FILE, index_col=0)
ref_df['slide_id'] = ref_df['wsi_file_name'].apply(lambda x: x.split('.')[0])
rna_cols     = [c for c in ref_df.columns if c.startswith('rna_')]
ref_genes    = [c.replace('rna_', '') for c in rna_cols]
common_genes = [g for g in gene_list if g in ref_genes]
common_idx   = [gene_list.index(g) for g in common_genes]
bulk_cols    = ['rna_' + g for g in common_genes]
G = len(common_genes)

train_embs = F.normalize(torch.tensor(
    np.load(f'{FT_EMB}/train_img_embs.npy'), dtype=torch.float32, device=device), dim=-1)
train_expr = torch.tensor(
    np.load(f'{FT_EMB}/train_exprs.npy'), dtype=torch.float32, device=device)

matched = [(row['slide_id'], row[bulk_cols].values.astype(float))
           for _, row in ref_df.iterrows()
           if os.path.exists(f'{TCGA_EMB}/{row["slide_id"]}.npy')]
print(f'Slides: {len(matched)}, Genes: {G}')

print('Computing tile predictions...')
slide_tile_preds, slide_bulks = [], []

for sid, bulk in matched:
    embs = F.normalize(torch.tensor(
        np.load(f'{TCGA_EMB}/{sid}.npy'), dtype=torch.float32, device=device), dim=-1)
    with torch.no_grad():
        sim        = torch.clamp(embs @ train_embs.T, min=0)
        weights    = sim / (sim.sum(dim=1, keepdim=True) + 1e-8)
        tile_preds = (weights @ train_expr).cpu().numpy()
    slide_tile_preds.append(tile_preds[:, common_idx])
    slide_bulks.append(bulk)
    del embs
torch.cuda.empty_cache()

S        = len(slide_tile_preds)
bulk_arr = np.array(slide_bulks)
print(f'Done. Slides: {S}')

def calc_pcc(pred_arr, bulk_arr):
    gene_pccs = [pearsonr(pred_arr[:,j], bulk_arr[:,j])[0]
                 for j in range(pred_arr.shape[1])
                 if pred_arr[:,j].std() > 1e-8 and bulk_arr[:,j].std() > 1e-8]
    slide_pccs = [pearsonr(pred_arr[i], bulk_arr[i])[0]
                  for i in range(pred_arr.shape[0])
                  if pred_arr[i].std() > 1e-8 and bulk_arr[i].std() > 1e-8]
    return np.array(gene_pccs).mean(), np.array(slide_pccs).mean()

Ks = [50, 100, 200, 300, 500, 'all']

# Spearman은 벡터화로 빠르게
def compute_spearman_scores(tile_preds, bulk):
    # rank 변환 후 PCC = Spearman
    from scipy.stats import rankdata
    bulk_rank  = rankdata(bulk)
    tile_ranks = np.apply_along_axis(rankdata, 1, tile_preds)  # (T, G)
    # 각 tile의 rank vector와 bulk rank의 PCC
    bulk_rank_c  = bulk_rank - bulk_rank.mean()
    tile_ranks_c = tile_ranks - tile_ranks.mean(axis=1, keepdims=True)
    num   = (tile_ranks_c * bulk_rank_c).sum(axis=1)
    denom = np.sqrt((tile_ranks_c**2).sum(axis=1)) * np.sqrt((bulk_rank_c**2).sum())
    scores = np.where(denom > 1e-8, num / denom, -999)
    return scores

methods = ['PCC', 'Spearman', 'Cosine']
results = {m: [] for m in methods}

for method_name in methods:
    print(f'\n[{method_name}]')
    print(f'{"K":>6} | {"Gene-wise mean":>14} | {"Slide-wise mean":>15}')
    print('-' * 42)

    for K in Ks:
        preds = []
        for tile_preds, bulk in zip(slide_tile_preds, slide_bulks):
            T = len(tile_preds)

            if method_name == 'PCC':
                # 벡터화: mean 보정 후 내적
                bulk_c      = bulk - bulk.mean()
                tile_c      = tile_preds - tile_preds.mean(axis=1, keepdims=True)
                num         = (tile_c * bulk_c).sum(axis=1)
                denom       = np.sqrt((tile_c**2).sum(axis=1)) * np.sqrt((bulk_c**2).sum())
                scores      = np.where(denom > 1e-8, num / denom, -999)

            elif method_name == 'Spearman':
                scores = compute_spearman_scores(tile_preds, bulk)

            elif method_name == 'Cosine':
                tile_norm = F.normalize(
                    torch.tensor(tile_preds, dtype=torch.float32, device=device), dim=-1)
                bulk_norm = F.normalize(
                    torch.tensor(bulk, dtype=torch.float32, device=device).unsqueeze(0), dim=-1)
                scores = (tile_norm @ bulk_norm.T).squeeze().cpu().numpy()

            if K == 'all':
                selected = tile_preds
            else:
                k        = min(K, T)
                valid    = scores > -999
                v_idx    = np.where(valid)[0]
                top_idx  = v_idx[np.argsort(scores[valid])[::-1][:k]]
                selected = tile_preds[top_idx]

            preds.append(selected.sum(axis=0))

        pred_arr       = np.array(preds)
        g_mean, s_mean = calc_pcc(pred_arr, bulk_arr)
        results[method_name].append((K, g_mean, s_mean))
        print(f'{str(K):>6} | {g_mean:>14.4f} | {s_mean:>15.4f}')

# 최종 비교
print('\n' + '='*50)
print('최종 비교 (K=100)')
print('='*50)
print(f'{"방법":<12} | {"Gene-wise":>10} | {"Slide-wise":>11}')
print('-' * 38)
for m in methods:
    r = {k: (g, s) for k, g, s in results[m]}
    print(f'{m:<12} | {r[100][0]:>10.4f} | {r[100][1]:>11.4f}')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Tile Ranking Metric Comparison (fold_03)', fontsize=13)
colors = {'PCC': '#2ecc71', 'Spearman': '#3498db', 'Cosine': '#e67e22'}
K_nums = [50, 100, 200, 300, 500]

for m in methods:
    r      = {k: (g, s) for k, g, s in results[m]}
    g_vals = [r[k][0] for k in K_nums]
    s_vals = [r[k][1] for k in K_nums]
    axes[0].plot(K_nums, g_vals, '-o', color=colors[m], label=m, linewidth=2, markersize=6)
    axes[1].plot(K_nums, s_vals, '-o', color=colors[m], label=m, linewidth=2, markersize=6)

for ax, title, ylabel in zip(axes,
    ['Gene-wise PCC vs K', 'Slide-wise PCC vs K'],
    ['Gene-wise PCC (mean)', 'Slide-wise PCC (mean)']):
    ax.set_xlabel('K (top tile count)', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.axhline(y=0, color='black', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.savefig(f'{OUT_DIR}/metric_comparison_3methods.png', dpi=150, bbox_inches='tight')
plt.close()
print(f'\nSaved: {OUT_DIR}/metric_comparison_3methods.png')
print('Done!')
EOF

In [1]:
# HDF5 파일에서 실제 tile 크기 확인
python3 -c "
import h5py
import numpy as np

hdf5_path = '/project_antwerp/TCGA-HNSC/TCGA_patch/TCGA-CQ-7068-01Z-00-DX1/TCGA-CQ-7068-01Z-00-DX1.hdf5'
with h5py.File(hdf5_path, 'r') as f:
    keys = list(f.keys())
    print(f'Total tiles: {len(keys)}')
    print(f'Sample keys: {keys[:5]}')
    
    # 첫 번째 tile 크기 확인
    first_tile = f[keys[0]][:]
    print(f'Tile shape: {first_tile.shape}')
    print(f'Dtype: {first_tile.dtype}')
    
    # 몇 개 더 확인
    for k in keys[:3]:
        print(f'  {k}: {f[k][:].shape}')
"

SyntaxError: unterminated string literal (detected at line 2) (2804515573.py, line 2)